In [4]:
import duckdb
import os
import time

# Setup paths
VOCAB_DIR = "../data/omop_vocab"
DB_PATH = "../data/omop_clinical.duckdb"

def load_concept_table():
    """
    Connects to DuckDB and loads the OMOP CONCEPT table from the Athena CSV.
    DuckDB is highly optimized for reading large CSVs directly.
    """
    concept_csv = os.path.join(VOCAB_DIR, "CONCEPT.csv")
    
    if not os.path.exists(concept_csv):
        print(f"❌ Error: The file {concept_csv} does not exist.")
        return

    print("🔌 Connecting to DuckDB...")
    
    try:
        # Open connection (safely closed in the finally block)
        con = duckdb.connect(DB_PATH)
        
        print("⏳ Loading CONCEPT table... (DuckDB is fast, but this might take a few seconds)")
        start_time = time.time()
        
        # Drop the table if it was partially loaded in a previous run
        con.execute("DROP TABLE IF EXISTS concept")
        
        # Load data directly from CSV to the disk-based database
        con.execute(f"""
            CREATE TABLE concept AS 
            SELECT * FROM read_csv_auto('{concept_csv}', header=True, delim='\t', nullstr='', sample_size=100000)
        """)
        
        elapsed_time = time.time() - start_time
        
        # Count rows without loading into Pandas to prevent memory issues
        count = con.execute("SELECT COUNT(*) FROM concept").fetchone()[0]
        print(f"✅ Success! Loaded {count:,} concepts in {elapsed_time:.2f} seconds.")
        
        # Show 5 results natively (without using fetchdf)
        print("\n🔎 Sample of loaded SNOMED clinical findings:")
        sample = con.execute("""
            SELECT concept_id, concept_name, concept_class_id 
            FROM concept 
            WHERE vocabulary_id = 'SNOMED' AND concept_class_id = 'Clinical Finding'
            LIMIT 5
        """).fetchall()
        
        for row in sample:
            print(f" - ID: {row[0]:<10} | Name: {row[1]:<30} | Class: {row[2]}")
            
    except Exception as e:
        print(f"❌ Critical error during load: {e}")
        
    finally:
        # Golden rule: ALWAYS close the connection
        con.close()
        print("\n🔒 DuckDB connection closed safely.")

# Execute the loader
load_concept_table()

🔌 Connecting to DuckDB...
⏳ Loading CONCEPT table... (DuckDB is fast, but this might take a few seconds)
✅ Success! Loaded 6,456,570 concepts in 15.05 seconds.

🔎 Sample of loaded SNOMED clinical findings:
 - ID: 42538812   | Name: Somatic hallucination          | Class: Clinical Finding
 - ID: 40629514   | Name: Stillbirth                     | Class: Clinical Finding
 - ID: 40277853   | Name: Small visual image             | Class: Clinical Finding
 - ID: 40304440   | Name: Pain: [site of GIT] or [abdominal site symptom] or [flank] or [subcostal] or [iliac fossa] | Class: Clinical Finding
 - ID: 40304969   | Name: Micturition stream normal      | Class: Clinical Finding

🔒 DuckDB connection closed safely.
